# Training a SimpleNN model

## Imports and dependencies

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

import torch
import torch.nn as nn

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from hordesofhell.models.simple_dataloader import create_simple_dataloaders
from hordesofhell.models.simple_nn import SimpleNN

csv_path = PROJECT_ROOT / "simple_output.csv"
train_loader, val_loader, meta = create_simple_dataloaders(
    csv_path=csv_path,
    batch_size=16,
    train_split=0.8,
    seed=42,
)

## Training Loop

In [ ]:
model = SimpleNN(input_features=meta["input_features"])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 2500

meta_training_data = {}
best_val_loss = float("inf")
best_index = -1
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)

    val_loss /= len(val_loader.dataset)
    print(f"Epoch {epoch + 1:02d}/{epochs} | train_loss={train_loss:.2f} | val_loss={val_loss:.2f}")
    if epoch == 0:
        print(f"Initial best_val_loss={best_val_loss:.2f}")
        best_index = epoch
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_index = epoch
        torch.save(model.state_dict(), PROJECT_ROOT / "best_model.pth")
        print(f"Best validation loss: {best_val_loss:.2f} at epoch {best_index + 1:02d}")

    meta_training_data[epoch] = {
        "train_loss": train_loss,
        "val_loss": val_loss,
        "best_val_loss": best_val_loss
    }

## Graphs

In [ ]:
df_meta_training = pd.DataFrame.from_dict(meta_training_data, orient="index")
df_meta_training.head()

In [ ]:
df_meta_training.describe()

In [ ]:
sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_meta_training[["train_loss", "val_loss", "best_val_loss"]])
plt.title("Train vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()